In [ ]:
%%writefile submission.py
# ！！！PLEASE DO NOT EDIT IT ！！！
# This command will save the code you wrote as a submission.py file in the same directory, for the test script to read.

import heapq
import json
import mmap
from collections import defaultdict
from collections.abc import Mapping, Sequence
from concurrent.futures import Future, ThreadPoolExecutor
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Final, NamedTuple, Protocol, cast

import regex

# trainer.py

"""Byte-level BPE (BBPE) trainer implementation.

This module provides a scalable implementation of Byte-level Byte Pair Encoding (BBPE)
for training tokenizers on large text corpora.
"""

@dataclass
class BBPETrainerConfig:
    """Configuration of a BBPE trainer.

    Attributes:
        vocab_size: Target vocabulary size, including special tokens.
        min_frequency: Minimum pair frequency for a merge to be considered.
        max_workers: Maximum number of worker threads for parallel I/O.
        chunk_size_bytes: Logical chunk size when splitting large corpora.
        seed: Random seed (unused, kept for compatibility).
        special_tokens: List of special tokens that must appear in the
            vocabulary and never be merged away.
    """

    vocab_size: int = 32000
    min_frequency: int = 2
    max_workers: int = 8
    chunk_size_bytes: int = 8 * 1024 * 1024
    seed: int = 42
    special_tokens: Sequence[str] = field(
        default_factory=lambda: ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
    )


class BBPEModel:
    """Container for a trained BBPE model."""

    def __init__(
        self,
        vocab: Mapping[bytes, int],
        merges: Sequence[tuple[bytes, bytes]],
        special_tokens: Sequence[str],
    ) -> None:
        self.vocab: dict[bytes, int] = dict(vocab)
        self.merges: list[tuple[bytes, bytes]] = list(merges)
        self.special_tokens: list[str] = list(special_tokens)


class BBPETrainer:
    """Byte-level BPE (BBPE) trainer."""

    def __init__(self, config: BBPETrainerConfig | None = None) -> None:
        self.config: BBPETrainerConfig = config or BBPETrainerConfig()
        self._vocab: dict[bytes, int] = {}
        self._merges: list[tuple[bytes, bytes]] = []

    def train(self, files: Sequence[str | Path]) -> BBPEModel:
        """Train a BBPE model from one or more text files.

        Args:
            files: Paths to input text files (UTF-8 encoded).

        Returns:
            A BBPEModel instance containing the learned vocabulary and merges.
        """
        if not files:
            raise ValueError("At least one file must be provided")

        path_files = [Path(f) if isinstance(f, str) else f for f in files]

        # Stage 1: Preprocess corpus into sequences
        sequences = list(self._preprocess_corpus(path_files))

        # Handle empty corpus
        if not sequences:
            vocab = self._init_base_vocab()
            self._vocab = vocab
            self._merges = []
            return BBPEModel(vocab=vocab, merges=[], special_tokens=list(self.config.special_tokens))

        # Stage 2: Run merge loop
        vocab, merges = self._merge_loop(sequences)
        self._vocab = vocab
        self._merges = merges

        return BBPEModel(vocab=vocab, merges=merges, special_tokens=list(self.config.special_tokens))

    def save(self, output_dir: str | Path) -> None:
        """Persist the trained model to disk."""
        if not self._vocab:
            raise ValueError("Model has not been trained yet. Call train() first.")

        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)

        # Save vocabulary as JSON
        vocab_file = output_path / "vocab.json"
        vocab_str = {token_bytes.decode('latin-1'): idx for token_bytes, idx in self._vocab.items()}
        with open(vocab_file, 'w', encoding='utf-8') as f:
            json.dump(vocab_str, f, ensure_ascii=False, indent=2)

        # Save merges as text file
        merges_file = output_path / "merges.txt"
        with open(merges_file, 'w', encoding='utf-8') as f:
            for token1, token2 in self._merges:
                _ = f.write(f"{token1.decode('latin-1')} {token2.decode('latin-1')}\n")

        # Save special tokens
        special_tokens_file = output_path / "special_tokens.json"
        with open(special_tokens_file, 'w', encoding='utf-8') as f:
            json.dump(list(self.config.special_tokens), f, ensure_ascii=False, indent=2)

    def _init_base_vocab(self) -> dict[bytes, int]:
        """Initialize vocabulary with 256 bytes + special tokens."""
        vocab: dict[bytes, int] = {}
        next_id = 0

        for byte_val in range(256):
            vocab[bytes([byte_val])] = next_id
            next_id += 1

        for special_token in self.config.special_tokens:
            token_bytes = special_token.encode('utf-8')
            if token_bytes not in vocab:
                vocab[token_bytes] = next_id
                next_id += 1

        return vocab

    def _preprocess_corpus(self, files: Sequence[Path]) -> list[list[int]]:
        """Preprocess corpus files into byte sequences."""
        GPT2_PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        special_tokens_pattern: regex.Pattern[str] | None = None
        if self.config.special_tokens:
            sorted_tokens = sorted(self.config.special_tokens, key=len, reverse=True)
            escaped = [regex.escape(token) for token in sorted_tokens]
            special_tokens_pattern = regex.compile("|".join(escaped))

        def find_utf8_boundary(data: bytes, pos: int) -> int:
            if pos >= len(data):
                return len(data)
            while pos > 0 and (data[pos] & 0b11000000) == 0b10000000:
                pos -= 1
            return pos

        def process_chunk(file_path: Path, start: int, end: int) -> list[list[int]]:
            with open(file_path, 'rb') as f:
                if end - start < 1024:
                    _ = f.seek(start)
                    chunk_data = f.read(end - start)
                else:
                    with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
                        chunk_data = mm[start:end]

            try:
                text = chunk_data.decode('utf-8')
            except UnicodeDecodeError as e:
                raise ValueError(
                    f"File {file_path} contains invalid UTF-8 at position {start + e.start}."
                ) from e

            # Match tokenizer semantics: split on special tokens first so GPT-2
            # pretokenization does not consume special-token prefixes like " <|".
            text_parts = (
                special_tokens_pattern.split(text)
                if special_tokens_pattern is not None
                else [text]
            )

            pretokens: list[list[int]] = []
            for part in text_parts:
                if not part:
                    continue
                pretokens.extend(
                    list(token.encode("utf-8"))
                    for token in regex.findall(GPT2_PAT, part)
                    if token
                )

            return pretokens

        def get_chunks(file_path: Path) -> list[tuple[int, int]]:
            file_size = file_path.stat().st_size
            if file_size == 0:
                return []
            if file_size <= self.config.chunk_size_bytes:
                return [(0, file_size)]

            chunks: list[tuple[int, int]] = []
            start = 0
            with open(file_path, 'rb') as f:
                with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
                    while start < file_size:
                        tentative_end = min(start + self.config.chunk_size_bytes, file_size)
                        if tentative_end < file_size:
                            boundary_start = max(0, tentative_end - 4)
                            boundary_data = mm[boundary_start:tentative_end + 1]
                            local_pos = tentative_end - boundary_start
                            adjusted = find_utf8_boundary(boundary_data, local_pos)
                            actual_end = boundary_start + adjusted
                        else:
                            actual_end = file_size
                        if actual_end > start:
                            chunks.append((start, actual_end))
                            start = actual_end
                        else:
                            start += 1
            return chunks

        all_sequences: list[list[int]] = []
        with ThreadPoolExecutor(max_workers=self.config.max_workers) as executor:
            futures: list[Future[list[list[int]]]] = []
            for file_path in files:
                if not file_path.exists():
                    raise FileNotFoundError(f"File not found: {file_path}")
                for start, end in get_chunks(file_path):
                    futures.append(executor.submit(process_chunk, file_path, start, end))

            for future in futures:
                for seq in future.result():
                    if seq:
                        all_sequences.append(seq)

        return all_sequences

    def _merge_loop(self, sequences: list[list[int]]) -> tuple[dict[bytes, int], list[tuple[bytes, bytes]]]:
        """Run the BPE merge loop with fully incremental updates."""
        vocab = self._init_base_vocab()
        next_id = len(vocab)

        # Build word frequency dict
        word_freq: dict[tuple[bytes, ...], int] = defaultdict(int)
        for seq in sequences:
            word_tuple = tuple(bytes([b]) for b in seq)
            word_freq[word_tuple] += 1

        # Compute initial pair counts and pair-to-words index
        pair_counts: dict[tuple[bytes, bytes], int] = defaultdict(int)
        pair_to_words: dict[tuple[bytes, bytes], set[tuple[bytes, ...]]] = defaultdict(set)

        for word_tuple, freq in word_freq.items():
            for j in range(len(word_tuple) - 1):
                pair = (word_tuple[j], word_tuple[j + 1])
                pair_counts[pair] += freq
                pair_to_words[pair].add(word_tuple)

        # Calculate number of merges
        num_merges = max(0, self.config.vocab_size - len(vocab))
        merges: list[tuple[bytes, bytes]] = []

        for _ in range(num_merges):
            if not pair_counts:
                break

            # Find best pair (highest freq, lexicographically largest for ties)
            best_pair = max(pair_counts.items(), key=lambda x: (x[1], x[0]))[0]
            if pair_counts[best_pair] < self.config.min_frequency:
                break

            p0, p1 = best_pair
            merged = p0 + p1

            # Only process words that contain the best pair
            affected_words = list(pair_to_words.get(best_pair, set()))

            for word_tuple in affected_words:
                freq = word_freq.get(word_tuple, 0)
                if freq == 0:
                    continue

                # Remove word from word_freq
                del word_freq[word_tuple]

                # Decrement old pair counts and remove from index
                for j in range(len(word_tuple) - 1):
                    old_pair = (word_tuple[j], word_tuple[j + 1])
                    pair_counts[old_pair] -= freq
                    if pair_counts[old_pair] <= 0:
                        del pair_counts[old_pair]
                        if old_pair in pair_to_words:
                            del pair_to_words[old_pair]
                    else:
                        pair_to_words[old_pair].discard(word_tuple)

                # Apply merge
                new_word: list[bytes] = []
                j = 0
                while j < len(word_tuple):
                    if j < len(word_tuple) - 1 and word_tuple[j] == p0 and word_tuple[j + 1] == p1:
                        new_word.append(merged)
                        j += 2
                    else:
                        new_word.append(word_tuple[j])
                        j += 1
                new_tuple = tuple(new_word)

                # Add new word to word_freq
                word_freq[new_tuple] = word_freq.get(new_tuple, 0) + freq

                # Increment new pair counts and add to index
                for j in range(len(new_tuple) - 1):
                    new_pair = (new_tuple[j], new_tuple[j + 1])
                    pair_counts[new_pair] += freq
                    pair_to_words[new_pair].add(new_tuple)

            merges.append(best_pair)

            if merged not in vocab:
                vocab[merged] = next_id
                next_id += 1

        return vocab, merges


# ===============================================================================

# tokenizer.py

"""Byte-level BPE (BBPE) tokenizer implementation.

This module provides a tokenizer that uses trained BBPE models for encoding
and decoding text. Optimized for performance with heap-based merging and caching.
"""


class _CacheInfo(NamedTuple):
    """Type for lru_cache.cache_info() return value."""
    hits: int
    misses: int
    maxsize: int | None
    currsize: int


class _CachedWordEncoder(Protocol):
    """Protocol for lru_cache wrapped word encoder function."""
    
    def __call__(self, word: str) -> tuple[int, ...]: ...
    def cache_clear(self) -> None: ...
    def cache_info(self) -> _CacheInfo: ...


class BBPETokenizer:
    """Byte-level BPE tokenizer with optimized encoding.
    
    Performance optimizations:
    - Heap-based merge selection: O(n log n) instead of O(n²)
    - LRU cache for repeated word encodings
    - Linked-list style traversal to avoid list slicing
    """

    # GPT-2 style pre-tokenization pattern
    _GPT2_PAT: Final[str] = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    
    # Cache size for word encodings (tune based on memory constraints)
    _CACHE_SIZE: Final[int] = 8192

    def __init__(
        self,
        vocab: dict[bytes, int] | None = None,
        merges: list[tuple[bytes, bytes]] | None = None,
        special_tokens: list[str] | None = None,
    ) -> None:
        """Initialize tokenizer with optional pre-loaded model.

        Args:
            vocab: Mapping from token bytes to token IDs.
            merges: List of merge operations in order.
            special_tokens: List of special tokens.
        """
        self._vocab: dict[bytes, int] = vocab or {}
        self._vocab_inv: dict[int, bytes] = {v: k for k, v in self._vocab.items()}
        self._merges: list[tuple[bytes, bytes]] = merges or []
        self._special_tokens: list[str] = special_tokens or []
        self._special_tokens_set: frozenset[str] = frozenset(self._special_tokens)
        
        # Type annotations for patterns (assigned in _compile_pattern)
        self._special_pattern: regex.Pattern[str] | None
        self._pattern: regex.Pattern[str]

        # Build merge rank lookup for fast merging
        self._merge_ranks: dict[tuple[bytes, bytes], int] = {
            pair: i for i, pair in enumerate(self._merges)
        }

        # Compile pre-tokenization pattern
        self._compile_pattern()
        
        # Create cached version of _encode_word_impl
        # Cast to Protocol type for proper type checking of cache methods
        self._encode_word_cached: _CachedWordEncoder = cast(
            _CachedWordEncoder,
            cast(object, lru_cache(maxsize=self._CACHE_SIZE)(self._encode_word_impl))
        )

    def _compile_pattern(self) -> None:
        r"""Compile the pre-tokenization regex patterns.
        
        Special tokens are handled separately to avoid conflicts with GPT-2
        pattern's punctuation matching (` ?[^\s\p{L}\p{N}]+`).
        """
        # Compile GPT-2 pattern (always used for non-special text)
        self._pattern = regex.compile(self._GPT2_PAT)
        
        # Compile separate pattern for special tokens (sorted by length, longest first)
        if self._special_tokens:
            # Sort by length descending to match longer tokens first
            sorted_tokens = sorted(self._special_tokens, key=len, reverse=True)
            escaped = [regex.escape(t) for t in sorted_tokens]
            self._special_pattern = regex.compile(f"({'|'.join(escaped)})")
        else:
            self._special_pattern = None

    @classmethod
    def from_file(cls, model_dir: str | Path) -> BBPETokenizer:
        """Load a tokenizer from a saved model directory.

        Args:
            model_dir: Path to directory containing vocab.json and merges.txt.

        Returns:
            A BBPETokenizer instance.
        """
        model_path = Path(model_dir)

        # Load vocabulary
        vocab_file = model_path / "vocab.json"
        with open(vocab_file, encoding="utf-8") as f:
            vocab_str = cast(dict[str, int], json.load(f))

        # Convert string keys back to bytes using latin-1
        vocab: dict[bytes, int] = {
            k.encode("latin-1"): v for k, v in vocab_str.items()
        }

        # Load merges
        merges_file = model_path / "merges.txt"
        merges: list[tuple[bytes, bytes]] = []
        with open(merges_file, encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if not line:
                    continue
                # Split on first space only (tokens may contain spaces)
                parts = line.split(" ", 1)
                if len(parts) == 2:
                    token1 = parts[0].encode("latin-1")
                    token2 = parts[1].encode("latin-1")
                    merges.append((token1, token2))

        # Load special tokens if available
        special_tokens_file = model_path / "special_tokens.json"
        special_tokens: list[str] = []
        if special_tokens_file.exists():
            with open(special_tokens_file, encoding="utf-8") as f:
                special_tokens = cast(list[str], json.load(f))

        return cls(vocab=vocab, merges=merges, special_tokens=special_tokens)

    def encode(self, text: str) -> list[int]:
        """Encode text into token IDs.

        Args:
            text: Input text to encode.

        Returns:
            List of token IDs.
        """
        if not text:
            return []

        token_ids: list[int] = []
        vocab = self._vocab
        special_tokens_set = self._special_tokens_set
        
        # If we have special tokens, split text by them first
        if self._special_pattern is not None:
            # Split text while keeping the special tokens as separate items
            parts = self._special_pattern.split(text)
            
            for part in parts:
                if not part:
                    continue
                    
                if part in special_tokens_set:
                    # This is a special token - encode directly
                    token_bytes = part.encode("utf-8")
                    if token_bytes in vocab:
                        token_ids.append(vocab[token_bytes])
                else:
                    # Regular text - apply GPT-2 pre-tokenization
                    for pretoken in cast(list[str], self._pattern.findall(part)):
                        ids = self._encode_word_cached(pretoken)
                        token_ids.extend(ids)
        else:
            # No special tokens - just apply GPT-2 pre-tokenization
            for pretoken in cast(list[str], self._pattern.findall(text)):
                ids = self._encode_word_cached(pretoken)
                token_ids.extend(ids)

        return token_ids

    def _encode_word_impl(self, word: str) -> tuple[int, ...]:
        """Encode a single pre-tokenized word using optimized BPE.
        
        Uses a heap-based algorithm for O(n log n) complexity instead of O(n²).
        Returns tuple for hashability (caching).

        Args:
            word: A single pre-tokenized word.

        Returns:
            Tuple of token IDs for this word.
        """
        word_bytes = word.encode("utf-8")
        n = len(word_bytes)
        
        if n == 0:
            return ()
        
        if n == 1:
            # Single byte - direct lookup
            token = bytes([word_bytes[0]])
            return (self._vocab.get(token, self._vocab.get(b"[UNK]", 0)),)
        
        # Initialize tokens and linked-list structure
        # tokens[i] = current token at position i (None if merged into previous)
        tokens: list[bytes | None] = [bytes([b]) for b in word_bytes]
        
        # next_pos[i] = next active position after i (-1 if end)
        # prev_pos[i] = previous active position before i (-1 if start)
        next_pos: list[int] = [i + 1 for i in range(n)]
        next_pos[n - 1] = -1
        prev_pos: list[int] = [i - 1 for i in range(n)]
        
        merge_ranks = self._merge_ranks
        
        # Build initial heap: (rank, position, left_token, right_token)
        # We include tokens in heap entries to detect stale entries
        heap: list[tuple[int, int, bytes, bytes]] = []
        
        pos = 0
        while pos != -1:
            next_p = next_pos[pos]
            if next_p != -1:
                left = tokens[pos]
                right = tokens[next_p]
                if left is not None and right is not None:
                    pair = (left, right)
                    if pair in merge_ranks:
                        heapq.heappush(heap, (merge_ranks[pair], pos, left, right))
            pos = next_p
        
        # Process merges
        while heap:
            _rank, pos, left_expected, right_expected = heapq.heappop(heap)
            
            # Check if this entry is stale (tokens have changed)
            if tokens[pos] is None:
                continue
            next_p = next_pos[pos]
            if next_p == -1:
                continue
            if tokens[next_p] is None:
                continue
            
            left = tokens[pos]
            right = tokens[next_p]
            
            # Verify tokens haven't changed since heap entry was created
            if left != left_expected or right != right_expected:
                continue
            
            # Perform merge (left and right are verified non-None above)
            assert left is not None and right is not None
            merged = left + right
            tokens[pos] = merged
            tokens[next_p] = None
            
            # Update linked list: remove next_p from chain
            next_next = next_pos[next_p]
            next_pos[pos] = next_next
            if next_next != -1:
                prev_pos[next_next] = pos
            
            # Add new pairs to heap
            # New pair with previous token
            prev_p = prev_pos[pos]
            if prev_p != -1:
                prev_token = tokens[prev_p]
                if prev_token is not None:
                    pair = (prev_token, merged)
                    if pair in merge_ranks:
                        heapq.heappush(heap, (merge_ranks[pair], prev_p, prev_token, merged))
            
            # New pair with next token
            if next_next != -1:
                next_token = tokens[next_next]
                if next_token is not None:
                    pair = (merged, next_token)
                    if pair in merge_ranks:
                        heapq.heappush(heap, (merge_ranks[pair], pos, merged, next_token))
        
        # Collect final tokens
        vocab = self._vocab
        unk_id = vocab.get(b"[UNK]", 0)
        
        result: list[int] = []
        pos = 0
        while pos != -1:
            token = tokens[pos]
            if token is not None:
                result.append(vocab.get(token, unk_id))
            pos = next_pos[pos]
        
        return tuple(result)

    def _encode_word(self, word: str) -> list[int]:
        """Encode a single pre-tokenized word using BPE.

        This is a wrapper that returns a list (for API compatibility).

        Args:
            word: A single pre-tokenized word.

        Returns:
            List of token IDs for this word.
        """
        return list(self._encode_word_cached(word))

    def decode(self, ids: Sequence[int]) -> str:
        """Decode token IDs back to text.

        Args:
            ids: Sequence of token IDs.

        Returns:
            Decoded text string.
        """
        if not ids:
            return ""

        # Convert IDs to bytes
        vocab_inv = self._vocab_inv
        byte_chunks: list[bytes] = [
            vocab_inv[token_id]
            for token_id in ids
            if token_id in vocab_inv
        ]

        # Concatenate and decode
        all_bytes = b"".join(byte_chunks)
        try:
            return all_bytes.decode("utf-8")
        except UnicodeDecodeError:
            # Fallback: decode with error replacement
            return all_bytes.decode("utf-8", errors="replace")

    def encode_batch(self, texts: Sequence[str]) -> list[list[int]]:
        """Encode multiple texts into token IDs.

        Args:
            texts: Sequence of input texts.

        Returns:
            List of token ID lists.
        """
        return [self.encode(text) for text in texts]

    def decode_batch(self, ids_batch: Sequence[Sequence[int]]) -> list[str]:
        """Decode multiple token ID sequences back to text.

        Args:
            ids_batch: Sequence of token ID sequences.

        Returns:
            List of decoded text strings.
        """
        return [self.decode(ids) for ids in ids_batch]

    @property
    def vocab_size(self) -> int:
        """Return the vocabulary size."""
        return len(self._vocab)

    @property
    def special_tokens(self) -> list[str]:
        """Return the list of special tokens."""
        return self._special_tokens.copy()

    def get_vocab(self) -> dict[str, int]:
        """Return the vocabulary as a string-to-ID mapping.

        Returns:
            Dictionary mapping token strings to IDs.
        """
        return {k.decode("latin-1"): v for k, v in self._vocab.items()}
    
    def clear_cache(self) -> None:
        """Clear the word encoding cache."""
        self._encode_word_cached.cache_clear()
    
    def cache_info(self) -> str:
        """Return cache statistics."""
        info = self._encode_word_cached.cache_info()
        return f"hits={info.hits}, misses={info.misses}, size={info.currsize}/{info.maxsize}"



Overwriting submission.py


In [1]:
import sys
import os
import pytest

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
os.chdir(project_root)

# Please CHANGE submission_dir to the directory of your answer.ipynb file, below is an example
submission_dir = os.path.join(project_root, "answers/dreamonex")
# submission_dir = os.path.join(project_root, "answers/xxxx")
if submission_dir not in sys.path:
    sys.path.insert(0, submission_dir)

# RUN TEST
# Args description:
# "-v": Verbose output
# "tests": Points to the test directory
# "-k sink": Run only tests related to "sink attention" (optional, for speed)
args = [
    "-v",
    "tests", 
    "-k", "test_train_bpe"
]

print(f"Current Working Directory: {os.getcwd()}")
pytest.main(args)

Current Working Directory: /home/dreamonex/srcs/sosd-works
============================= test session starts ==============================
platform linux -- Python 3.14.4, pytest-9.0.3, pluggy-1.6.0 -- /home/dreamonex/srcs/sosd-works/.venv/bin/python
cachedir: .pytest_cache
rootdir: /home/dreamonex/srcs/sosd-works
plugins: jaxtyping-0.3.9
collecting ... collected 28 items / 25 deselected / 3 selected

tests/test_train_bpe.py::test_train_bpe_speed PASSED                     [ 33%]
tests/test_train_bpe.py::test_train_bpe PASSED                           [ 66%]
tests/test_train_bpe.py::test_train_bpe_special_tokens PASSED            [100%]

======================= 3 passed, 25 deselected in 3.34s =======================


<ExitCode.OK: 0>